In [ ]:
from pathlib import Path
import pandas as pd
from collections import defaultdict
import yaml
import numpy as np

def get_project_root():
    return Path().resolve().parent

# This file is the master mapping from city to province, the comprehensivre list with codes
# there are 514 unique districts/cities, and 38 unique provinces
df = pd.read_csv(get_project_root() / "data" / "mapping" / "master_2024.csv")

# These are the manually coded assessments of the index indicators (by Hana)
# otherwise known as Hana's dataset
# there are 509 unique cities here (5 missing are the DKI JAKARTA...)
policy_encoding = pd.read_csv(get_project_root() / "data" / "processed" / "policy_encoding.csv")

# maps from official (master) fullname to/from Hanas encoding
with open(get_project_root() / "data" / "mapping"/"city_mapping.yaml", "r") as f:
    city_mapping = yaml.safe_load(f)

city_mapping_reversed = {v: k for k, v in city_mapping.items()}

In [2]:
# create nested dict version of master mapping from master full name for quick reference
# dont need to run unless master changes
def create_tree_yaml(df):
    prov_tree = defaultdict(list)

    for _, row in df.iterrows():
        prov_tree[row["nama_prov"]].append(row["fullname"])

    output_path = get_project_root() / "data" / "mapping"
    output_path.mkdir(parents=True, exist_ok=True)
    file_path = output_path / "prov_tree.yaml"

    with open(file_path, "w", encoding="utf-8") as f:
        yaml.dump(
            prov_tree,
            f,
            allow_unicode=True,
            sort_keys=True,
            default_flow_style=False
        )

### Policy texts

In [11]:
# look at filenames and missing files in directory
# This is the raw data (policy) documents, not yet translated
base = Path.cwd().parent / "data" / "raw" / "localpolicies"
filenames = policy_encoding['filename'].dropna().unique()

txt_exists = np.array([(base / f).exists() for f in filenames])

pdf_exists = np.array([
    (base / Path(f).with_suffix(".pdf").name).exists()
    for f in filenames
])

print(f"Total expected files: {len(filenames)}")
print(f"TXT exists: {txt_exists.sum()}")
print(f"TXT missing but PDF exists: {((~txt_exists) & pdf_exists).sum()}")
print(f"Missing entirely: {((~txt_exists) & (~pdf_exists)).sum()}")

Total expected files: 198
TXT exists: 149
TXT missing but PDF exists: 6
Missing entirely: 43


In [15]:
policy_encoding[policy_encoding['policy_change']==1]

,fullname,year,policy_version,1.1,1.2,1.3,1.4,1.5,1.6,1.7,...,7.2,7.3,7.4,7.5,adoption_isu,vaw_reported_cases,gdp,policy_change,has_policy,filename
304,ACEH BIREUEN,2022,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,25.0,15470.0,1,1,ACEH_BIREUEN.txt
503,ACEH SUBULUSSALAM KOTA,2023,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,1.0,1.0,NaN,7.0,2560.0,1,1,ACEH_SUBULUSSALAM_KOTA.txt
515,BALI BADUNG,2013,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,NaN,31790.0,1,1,BALI_BADUNG.txt
540,BALI BANGLI,2016,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,NaN,5457.0,1,1,BALI_BANGLI.txt
565,BALI BULELENG,2019,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,18.0,35362.0,1,1,BALI_BULELENG.txt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10467,SUMATERA SELATAN PRABUMULIH KOTA,2021,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,30.0,8346.0,1,1,SUMATERA_SELATAN_PRABUMULIH_KOTA.txt
10506,SUMATERA UTARA BATU BARA,2016,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,NaN,27555.0,1,1,SUMATERA_UTARA_BATU_BARA.txt
10689,SUMATERA UTARA LABUHANBATU SELATAN,2023,1,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,NaN,6.0,37584.0,1,1,SUMATERA_UTARA_LABUHANBATU_SELATAN.txt
10929,SUMATERA UTARA PADANGSIDIMPUAN KOTA,2021,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,NaN,19.0,6784.0,1,1,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt


In [13]:
# specifically, which texts are missing?
missing_mask = (~txt_exists) & (~pdf_exists)
print("\nMissing files:")
for f in filenames[missing_mask]:
    print(f)


Missing files:
ACEH_SUBULUSSALAM_KOTA.txt
DI_YOGYAKARTA_BANTUL_v2.txt
DI_YOGYAKARTA_GUNUNGKIDUL_v2.txt
DKI_JAKARTA_KEPULAUAN_SERIBU.txt
JAMBI_SAROLANGUN.txt
JAWA_BARAT_BANDUNG_KOTA.txt
JAWA_BARAT_BEKASI.txt
JAWA_BARAT_BEKASI_KOTA_v2.txt
JAWA_BARAT_CIREBON_KOTA.txt
JAWA_BARAT_SUMEDANG.txt
JAWA_TENGAH_KARANGANYAR_v2.txt
JAWA_TENGAH_KEBUMEN.txt
JAWA_TENGAH_PURWOREJO.txt
JAWA_TENGAH_SEMARANG_KOTA_v2.txt
JAWA_TENGAH_SURAKARTA_KOTA.txt
KALIMANTAN_SELATAN_TANAH_BUMBU.txt
KALIMANTAN_TENGAH_KATINGAN.txt
KALIMANTAN_TENGAH_PULANG_PISAU.txt
KALIMANTAN_TENGAH_SUKAMARA.txt
KALIMANTAN_TIMUR_KUTAI_TIMUR.txt
KALIMANTAN_TIMUR_PENAJAM_PASER_UTARA.txt
KEPULAUAN_BANGKA_BELITUNG_BANGKA_BARAT_v2.txt
KEPULAUAN_BANGKA_BELITUNG_BANGKA_SELATAN_v2.txt
LAMPUNG_PESISIR_BARAT.txt
LAMPUNG_TANGGAMUS.txt
NUSA_TENGGARA_BARAT_DOMPU.txt
NUSA_TENGGARA_BARAT_LOMBOK_TENGAH_v2.txt
NUSA_TENGGARA_TIMUR_TIMOR_TENGAH_SELATAN.txt
RIAU_BENGKALIS.txt
RIAU_PEKANBARU_KOTA.txt
SULAWESI_BARAT_MAMUJU.txt
SULAWESI_SELATAN_BULUKUMBA.txt
S

# Q for Hana:

1.
Mapping: There are 5 missing "DKI JAKARTA" KOTAS
mapped only this one: "DKI JAKARTA KEPULAUAN SERIBU": "DKI Jakarta"
- DKI JAKARTA JAKARTA SELATAN KOTA
- DKI JAKARTA JAKARTA TIMUR KOTA
- DKI JAKARTA JAKARTA PUSAT KOTA
- DKI JAKARTA JAKARTA BARAT KOTA
- DKI JAKARTA JAKARTA UTARA KOTA

2.
"MALUKU KEPULAUAN TANIMBAR": "" **** renamed 2019?
"SULAWESI BARAT MAMUJU TENGAH": "Mamuju Tengah" ***** wrong provice categorisation?

3.
what happeend with updated documents? v2

4.
all of the above missing raw policy documents?

In [ ]:
# compare two raw sources xlsx